# This notebook show how to build a track

In [1]:
import sys
import os




project_path = r"C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder"
if project_path not in sys.path:
    sys.path.append(project_path)

import track_builder as tb




BASE_PATH = r"C:\Users\lamin\Documents\maitrise\ASTD\data"
YEAR      = 2019

MONTHS_TO_LOAD = [1, 2]

USECOLS   = "default"
SAMPLING  = [0, -1]

COLS_REQUIRED = [
    "shipid",
    "date_time_utc",
    "latitude",
    "longitude",
    "astd_cat",
    "flagname",
    "iceclass",
    "sizegroup_gt"
]


### Loading the first and last day of 5 month of data
-   create a variable `SAMPLING = [0, -1]`  to load only the first and last day

In [2]:
df_for_track = tb.load_astd_monthly(
    BASE_PATH, YEAR, months=MONTHS_TO_LOAD, progress=True,
    usecols=USECOLS, sampling=SAMPLING, remove_nan_rows=COLS_REQUIRED
)

c:\Users\lamin\miniconda3\envs\astd-track\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading ASTD CSVs:   0%|          | 0/2 [00:00<?, ?it/s]

Loading ASTD CSVs: 100%|██████████| 2/2 [02:41<00:00, 80.67s/it]


### Load all data for 5 month
- You need to precise that the `sampling = None` to load all data

In [3]:
df = tb.load_astd_monthly(
    BASE_PATH, YEAR, months=MONTHS_TO_LOAD, progress=True,
    usecols=USECOLS, sampling=None, remove_nan_rows=COLS_REQUIRED
)

Loading ASTD CSVs: 100%|██████████| 2/2 [02:30<00:00, 75.01s/it]


### Build the track table

In [4]:
tracks = tb.build_ship_tracks(df_for_track,
                              matching_strategy="balanced",
                            )

Data after cleaning:
  Date range: 2019-01-01 00:00:02+00:00 to 2019-02-28 23:59:58+00:00
  Ship types: ['other service offshore vessels' 'general cargo ships'
 'offshore supply ships' 'fishing vessels' 'passenger ships'
 'bulk carriers' 'other activities' 'ro-ro cargo ships' 'cruise ships'
 'refrigerated cargo ships' 'chemical tankers' 'container ships'
 'gas tankers' 'crude oil tankers' 'oil product tankers']
  Unique ships: 1873


C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:230: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['period_month'] = df['date_time_utc'].dt.to_period('M')


Creating segments for 2024 unique ship-months (segments)...
Successfully created 2024 monthly segments.
Sample segment: fishing vessels|iceland|fs ice class 1c|< 1000 gt in 2019-02


C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:382: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = (ship_means.groupby('astd_cat')


### Individual Track Inspection

In [8]:


# Filter for tracks containing more than one segment (i.e., successfully linked)
track_counts = tracks['track_id'].value_counts()
linked_tracks = track_counts[track_counts > 1].index.tolist()

if linked_tracks:
    target_track_id = linked_tracks[1] 
    print(f"Visualizing Track ID: {target_track_id}")

    # Load only the data need for the specific track_id : means loading only segments points for this track_id
    track_data_df = tb.load_track_data(track_ids=target_track_id, track_table=tracks, base_path=BASE_PATH, chunksize=500_000)
    
    fig = tb.plot_individual_track(
        track_id=target_track_id,
        track_table=tracks,
        astd_data=track_data_df,
        title=f"Reconstructed Trajectory | Track ID: {target_track_id}"
    )
    fig.show()
else:
    print("No multi-segment tracks found in the current sample.")

Visualizing Track ID: 599


Batch loading 1 tracks: 100%|██████████| 2/2 [00:56<00:00, 28.49s/it]

computing typical speeds...
Cleaning completed: 169 'ghost' or aberrant points removed.



C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:382: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:170: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



### You can use also the function `build_light_multi_track_data` to plot one specific track

In [9]:
work = tb.build_light_multi_track_data(track_table=tracks, specific_track_ids=[599], positions_df=df)

fig = tb.plot_ship_tracks(
    work,
    color_by="track_id",
    color_mode="categorical",
    show_points=False,
    map_style="open-street-map",
    title="Tracks (light multi-track sample)",
)
fig.update_layout(showlegend=False)
fig.show()

computing typical speeds...
Cleaning completed: 169 'ghost' or aberrant points removed.


C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:382: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:170: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\io\astd_loader.py:647: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups

## 2. Prepare Batch Data (Préparation par lots)
- `track_sampling=10` means: "Pick 10 random tracks for me"
- `point_stride=10` means: "Keep only 1 point every 10 points" (Optimization)

In [10]:
map_data = tb.build_light_multi_track_data(
    track_table=tracks,
    track_sampling=10,      # Try 50 or 100 if you want more!
    point_stride=10         
)


Batch loading 10 tracks: 100%|██████████| 2/2 [00:59<00:00, 29.87s/it]


computing typical speeds...
Cleaning completed: 206 'ghost' or aberrant points removed.


C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:170: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\io\astd_loader.py:647: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



### Batch Visualization (Visualisation Globale)
- We use `plot_ship_tracks` because we want to show all the track 

In [11]:
fig = tb.plot_ship_tracks(
    map_data,
    show_points=False,
    color_by="track_id",     # Give a unique color to each ship
    show_start_end=True,     # Show Start (Ring) and End (Dot) markers
    title="Fleet Overview: 10 Random Tracks"
)

fig.show()